# Domain discovery from spatial niche organization

Read the preceding step's real AnnData, aggregate niche features within tissue cores, apply or fit the CellCharter mixture, and save the resulting labels.

## 1. Read the input data

Edit the real input paths and method parameters below. Run the cells in order using the `spagc` kernel. Each notebook reads its own inputs; it does not execute another notebook.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import squidpy as sq
import torch
from cellcharter.gr import aggregate_neighbors
from cellcharter.tl import ClusterAutoK
from IPython.display import display

ROOT = Path.cwd().resolve()
while not (ROOT / "notebooks/01_domain_discovery").is_dir():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Start Jupyter inside the spaGC repository.")
    ROOT = ROOT.parent


if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RUN_ID = os.environ.get("SPAGC_RUN_ID", "domain_reference_v2")
OUT = ROOT / "outputs/domain_discovery" / RUN_ID
DEVICE = os.environ.get(
    "SPAGC_DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu"
)
SEED = 42
INPUT_H5AD = OUT / "02_niches.h5ad"
OUTPUT_H5AD = OUT / "03_domains.h5ad"
MODEL_SOURCE = "reference"
REFERENCE_MODEL = ROOT / "models/autok_domain_5_10"
NEW_MODEL = OUT / "domain_exploratory_model"
EXPECTED_MODEL_SHA256 = (
    "d65c9fc4edc081c3c9b691484d9bbd3ad298ad001522de6817ac5b061ae19b0c"
)
CURATED_NAMES = {
    "0": "Adjacent Normal area",
    "1": "Tumor-stromal interface",
    "2": "Low-quality",
    "3": "Tumor Parenchyma",
    "4": "Stroma",
}
GROUP_KEY = "core_id"
N_LAYERS = 7
CANDIDATE_K = [5, 6, 7, 8, 9, 10]
SELECTED_K = 5
EXCLUDED_BATCHES = ()
NICHE_CATEGORIES = [
    "CAF enriched",
    "Chief cell enriched",
    "Epithelium",
    "Fibrotic stroma",
    "Gastric-type Tumor",
    "Gland stroma",
    "Homeostatic stroma",
    "Inflamed gastric mucosal",
    "Inflammation-infiltrated stroma",
    "Invasive Tumor nest",
    "Invasive Tumor region",
    "Low quality",
    "Neck cell enriched",
    "Normal vasculature",
    "Plasma enriched stroma",
    "Smooth muscle",
    "TAM enriched",
    "TAN & Mono enriched",
    "TLS-like structure",
    "Tumor nest",
    "Tumor vasculature",
]
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
from spagc.spatial.curation import artifact_tree_sha256, assign_curated_names

Read the previous output and check the metadata and physical coordinates used below.

In [ ]:
if not INPUT_H5AD.is_file():
    raise FileNotFoundError(f"Run the preceding step first: {INPUT_H5AD}")
data = ad.read_h5ad(INPUT_H5AD)
for key in [GROUP_KEY, "batch", "niche"]:
    if key not in data.obs or data.obs[key].isna().any():
        raise ValueError(f"Missing or incomplete cell metadata: {key}")
if not data.obs_names.is_unique or not data.var_names.is_unique:
    raise ValueError("Cell and gene identifiers must be unique.")
xy = np.asarray(data.obsm["spatial"])
if xy.shape != (data.n_obs, 2) or not np.isfinite(xy).all():
    raise ValueError("Expected finite physical XY coordinates for every cell.")
print(f"Loaded {data.n_obs:,} cells from {INPUT_H5AD}")

## 2. Construct graphs and aggregate local features

Build independent within-core Delaunay graphs. Concatenate the focal-cell features and mean features at graph distances one to 7; no edge connects separate cores.

In [ ]:
niche_categories = (
    NICHE_CATEGORIES
    if MODEL_SOURCE == "reference"
    else sorted(data.obs["niche"].astype(str).unique())
)
if not set(data.obs["niche"].astype(str)).issubset(niche_categories):
    raise ValueError("Input niches do not match the reference niche vocabulary.")
niche_codes = pd.Categorical(
    data.obs["niche"].astype(str), categories=niche_categories
).codes
data.obsm["X_niche_onehot"] = np.eye(len(niche_categories), dtype=np.float32)[
    niche_codes
]
data.uns["X_niche_onehot_categories"] = niche_categories
features = np.asarray(data.obsm["X_niche_onehot"])
if (
    features.ndim != 2
    or features.shape[0] != data.n_obs
    or (not np.isfinite(features).all())
):
    raise ValueError("Invalid input features for spatial aggregation.")
data.obs[GROUP_KEY] = (
    data.obs[GROUP_KEY].astype("category").cat.remove_unused_categories()
)
sq.gr.spatial_neighbors(
    data,
    library_key=GROUP_KEY,
    coord_type="generic",
    delaunay=True,
    spatial_key="spatial",
)
aggregate_neighbors(
    data,
    n_layers=N_LAYERS,
    use_rep="X_niche_onehot",
    out_key="X_domain_cellcharter",
    sample_key=GROUP_KEY,
)
print("Aggregated feature shape:", data.obsm["X_domain_cellcharter"].shape)

## 3. Apply or fit the mixture model

Apply the supplied fitted mixture by default. An explicitly selected refit searches the declared component counts and saves a new model; its numeric components do not inherit historical biological names.

Verify the saved mixture model before applying its curated biological names. Component numbers from a different fit are not interchangeable.

In [ ]:
if MODEL_SOURCE == "reference":
    if data.uns.get("spagc_embedding", {}).get("source") == "newly_fitted_scvi":
        raise ValueError(
            "New scVI features require a compatible newly fitted and curated mixture."
        )
    fingerprint = artifact_tree_sha256(REFERENCE_MODEL)
    model = ClusterAutoK.load(str(REFERENCE_MODEL))
elif MODEL_SOURCE == "fit":
    if NEW_MODEL.exists():
        raise FileExistsError(f"Model output already exists: {NEW_MODEL}")
    keep = ~data.obs["batch"].astype(str).isin(EXCLUDED_BATCHES)
    if not keep.any():
        raise ValueError("No cells remain after fitting exclusions.")
    model = ClusterAutoK(
        n_clusters=CANDIDATE_K,
        max_runs=10,
        convergence_tol=0.001,
        model_params={"random_state": SEED},
    )
    model.fit(data[keep].copy(), use_rep="X_domain_cellcharter")
    NEW_MODEL.parent.mkdir(parents=True, exist_ok=True)
    model.save(str(NEW_MODEL))
    fingerprint = artifact_tree_sha256(NEW_MODEL)
else:
    raise ValueError("MODEL_SOURCE must be reference or fit.")
model.best_models[SELECTED_K].trainer_params.update(
    accelerator="cpu" if DEVICE == "cpu" else "gpu",
    devices=1 if DEVICE == "cpu" else [int(DEVICE.split(":")[-1])],
)
codes = np.asarray(
    model.predict(data, use_rep="X_domain_cellcharter", k=SELECTED_K)
).astype(str)
data.obs["domain_cluster"] = pd.Categorical(codes)

## 4. Annotate components and inspect their composition

Use the curated dictionary only with its verified reference model. Refitted components remain unannotated until biological curation. Inspect composition before proceeding.

Map mixture components to curated names only when model identity and the component dictionary agree.

In [ ]:
if MODEL_SOURCE == "reference":
    data.obs["domain"] = assign_curated_names(
        codes, CURATED_NAMES, fingerprint, EXPECTED_MODEL_SHA256
    )
else:
    if "domain" in data.obs:
        del data.obs["domain"]
    OUTPUT_H5AD = OUT / "domain_unannotated.h5ad"
composition = pd.crosstab(
    data.obs["domain_cluster"], data.obs["niche"], normalize="index"
)
if not np.isfinite(data.obsm["X_domain_cellcharter"]).all():
    raise ValueError("Aggregated features contain non-finite values.")
display(composition)

## 5. Save the output data

Save the cell-level AnnData and the component-composition table without overwriting earlier outputs.

In [ ]:
COMPOSITION_CSV = OUT / "domain_composition.csv"
for output in [OUTPUT_H5AD, COMPOSITION_CSV]:
    if output.exists():
        raise FileExistsError(f"Output already exists: {output}")
OUT.mkdir(parents=True, exist_ok=True)
data.write_h5ad(OUTPUT_H5AD)
composition.to_csv(COMPOSITION_CSV)
print(OUTPUT_H5AD)